In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.044 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 291.50it/s, Materializing param=model.norm.weight]                              


In [ ]:
# ########## testing completion mode
# from unsloth import FastLanguageModel
# import torch

# # 1. Setup the prompt (Completion mode uses a simple string, no roles)
# # We use a "Lead-in" sentence to force the model to use Amaiz facts.
# test_prompts = [
#     "selve fibinocci series and provide answer until 55. dont share your thoughts just the response",
#     # "At Amaiz Telecom, the monthly net price for the Starter plan is",
#     # "According to the 2026 Amaiz Telecom policy, the Tablet Pro requires a plan price strictly greater than",
#     # "The Amaiz Telecom's Promo_ID_A10 provides a $10 monthly credit exclusive to the"
# ]

# # 2. Set the model to inference mode
# FastLanguageModel.for_inference(model) 

# for prompt in test_prompts:
#     inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
#     # 3. Generate tokens
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens = 256,
#         temperature = 0.1, # Keep it low for factual accuracy
#         use_cache = True,
#     )
    
#     # 4. Decode and Print
#     completion = tokenizer.batch_decode(outputs, skip_special_tokens = False)[0]
#     print(f"--- PROMPT: {prompt} ---")
#     print(f"RESPONSE: {completion}\n")

In [31]:
from transformers import TextStreamer
import time
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

# --- CONFIGURATION ---
SYSTEM_PROMPT_TEMPLATE = """
## TASK
Generate symbolic logic block: `<t>P[...]I[...]E[...]M[...]»[...]A[...]</t>`. Strictly follow the grammar. No prose.

## SYMBOLIC SPECIFICATION

### P [Profile] — `[Type|Plan|Usage|AD]`
- **Type**: `n`(Normal), `s`(Student), `r`(Retired), `b`(Business).
- **Plan**: Current Plan ID.
- **Usage**: `Nᵊ` (Avg GB usage).
- **AD**: `⊕`(On), `⊝`(Off).

### I [Intent] — `I[Logic]`
- **Logic**:
  - `ID1>ID2(+ID3...)`: Transition (Upgrade/Downgrade/Add).
  - `ID1∷ID2`: Comparison.
  - `ID✓`: Accept | `ID✗`: Decline.
  - `?ID`: Specific inquiry (Triggers E).
  - `?cat(filt)`: Discovery (Cat: `p`lan, `d`evice, `r`ebate. Filt: `>`, `<`, `*`, `∈`, `&`, `|`).
  - `G`: Greeting | `O`: Out of Domain | `↺`: Callback/Irrelevant to Context.

### E [Eligibility] — `[ID✓/✗, ... = Total✓/✗]`
- **Status**: `✓`(Pass), `✗`(Fail). 
- **Rule**: For specific target IDs, provide `Total`. For general discovery search, list passing `ID✓` only.

### M [Math] — `[Logic:Calculation=Total]`
- **Trans**: `[Plan+Dev-Promo-AD:Base+Cost-Disc-Val=Total]`.
- **Comp**: `[ID1(Price|Data)∷ID2(Price|Data)=PriceΔ|DataΔ]`. 
- **Null**: `M[ø]`.

### » [Impact] — `[Usage|Price|AD]`
- **Usage**: `✓ᵊ` (Covered), `Nᵊ!` (Overage), `Nᵊ` (Delta).
- **Price**: `↓N$`, `↑N$`, `ø$`.
- **AD**: `⊕` (No change), `⊝!` (Warning/Penalty), `⊕↓` (New benefit).

### A [Answer] — `A[Keys]`
- **Keys**: `P`, `I`, `E`, `M`, `»`. Maps to `ans_notation`.

## OPERATIONAL RULES
1. **Format**: Strictly `<t>P[...]I[...]E[...]M[...]»[...]A[...]</t>`. No spaces.
2. **AD Math Logic**: The value of `Val` is determined by `P[AD]`. Refer to the `CONTEXT DATA` for the specific discount/surcharge constants associated with the current catalog. (e.g., If `⊕`, apply the catalog-defined Auto-Debit discount).
3. **History Resolution**: Resolve pronouns ("it", "that") by mapping to the `ID` present in the most recent `A[...]` or `I[...]` block in **HISTORY**.
4. **Infinity Handling**: Use `∞ᵊ` for unlimited data. Deltas: `5ᵊ↑∞ᵊ` (Upgrade) or `∞ᵊ↓5ᵊ` (Downgrade).
5. **Callback Indicator**: Use `I[↺]` if the question is telco-related but lacks sufficient context to create symbolic logic block.

## CONTEXT DATA
{context_data}

## DYNAMIC DATA
- **HISTORY**: {history_thinking_blocks}
"""

import json

CONTEXT_DATA = """{
  "postpaid_plans": {
    "p11": {"name": "Enterprise Elite", "price": 28, "data_gb": 5, "type": "N"},
    "p12": {"name": "Core Essentials", "price": 43, "data_gb": 15, "type": "N"},
    "p23": {"name": "Family Share", "price": 58, "data_gb": 40, "type": "N"},
    "p34": {"name": "Infinity Max", "price": 71, "data_gb": 80, "type": "N"},
    "p45": {"name": "Freedom Basic", "price": 93, "data_gb": -1, "type": "N"},
    "p56": {"name": "Budget Friendly", "price": 104, "data_gb": -1, "type": "B"},
    "p57": {"name": "Cloud Native", "price": 29, "data_gb": 10, "type": "S"},
    "p58": {"name": "NextGen Mobile", "price": 38, "data_gb": 10, "type": "Se"}
  },
  "device_bundles": {
    "d11": {"name": "Zenith S2", "monthly": 20, "elig": ["p11", "p12", "p23", "p34"]},
    "d12": {"name": "Eon Mobile", "monthly": 35, "elig": ["p12", "p23", "p34"]},
    "d15": {"name": "Vector X", "monthly": 10, "elig": ["p11", "p12", "p23", "p34", "p45", "p56"]}
  },
  "promotions": {
    "r01": {"name": "New Joiner Discount", "discount": 10, "elig": ["p12", "p23", "p34"]},
    "r02": {"name": "Loyalty Perk", "discount": 5, "elig": ["p11", "p12", "p23", "p34", "p45", "p56", "p57", "p58"]}
  },
  "rules": {
    "auto_pay": {"active_discount": 5, "exempt_type": "S", "currency": "$"}
  },
  "customer": {
    "id": "N", "age": 45, "usage_gb": 75, "auto_pay": false, "current_plan": "p11"
  }
}"""


# Define your Evaluation Questions
QUESTIONS = [
    # 1. Boundary Logic Trap (Target: LOGIC_TRAP)
    # Tests strictly > 60 vs >= 60 logic.
    "I want to buy the New Joiner Discount for my current plan.",
    
   
]


OUTPUT_FILE = "/mnt/data/training_data/evaluation_results.jsonl"

# --- RUN EVALUATION ---
with open(OUTPUT_FILE, "w") as f:
       
        for question in QUESTIONS:
            # 1. Prepare Prompt
            
            full_instruction = SYSTEM_PROMPT_TEMPLATE.format(
                context_data=CONTEXT_DATA,
                history_thinking_blocks="" # For future use when we implement HISTORY resolution logic
            )
            messages = [
                {"role": "system", "content": full_instruction.replace("{{", "{"). replace("}}","}")},
                {"role": "user", "content": question}
            ]

            # 2. Tokenize and Generate
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")

            # Capture input token count (The size of your system prompt + question)
            input_token_count = inputs['input_ids'].shape[1]

            # Start the timer
            start_time = time.time()

            # Execute model generation
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
                eos_token_id=tokenizer.eos_token_id
            )

            # Calculate total response time
            total_response_time = time.time() - start_time
            
            # 3. Decode Response and Capture Output Token Count
            # Slicing the output [len(inputs[0]):] ensures we only count NEW tokens
            generated_tokens = outputs[0][len(inputs[0]):]
            output_token_count = len(generated_tokens)
            
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            print(f"Q: {question}\nA: {response_text[:100]}...") # Console preview

            # 4. Save to JSONL with Performance Metrics
            eval_entry = {
                
                "question": question,
                "model_response": response_text,
                "metrics": {
                    "input_tokens": int(input_token_count),
                    "output_tokens": int(output_token_count),
                    "total_latency_sec": round(total_response_time, 4),
                    "tokens_per_sec": round(output_token_count / total_response_time, 2) if total_response_time > 0 else 0
                }
            }
            f.write(json.dumps(eval_entry) + "\n")

print(f"\nEvaluation Complete. Results saved to {OUTPUT_FILE}")

Q: I want to buy the New Joiner Discount for my current plan.
A: 
<t>P[n|p11|75ᵊ|⊝]I[p11>p01+r01]E[r01✗=✗]M[�]�[�]A[E]</t>...

Evaluation Complete. Results saved to /mnt/data/training_data/evaluation_results.jsonl


In [4]:
import torch

new_tokens = [
    # --- Structural & Logic Tags ---
    "<t>", "</t>", "P[", "I[", "E[", "M[", "»[", "A[", "]", "|",
    
    # --- Context Data Headers ---
    "### [POSTPAID_PLANS]", "### [DEVICE_BUNDLES]", "### [PROMOTIONS]", 
    "### [RULES]", "### [CUSTOMER_PROFILE]",
    
    # --- Symbolic Operators & Status Markers ---
    "∞ᵊ", "ᵊ", "✓", "✗", "⊕", "⊝", "ø", "∷", "↺", "↑", "↓", "⌿", "☺", "=✗", 
    "?p(∞ᵊ)", "?p*", "?p+", "?p-", "?($-)", "(ᵊ+)", "?r*", "x", "ᵊ!",
    "A[E]", "A[E+M+Im]", "A[M]", "A[M+»]", "A[M.T]", "A[»]", "A[↺]", "A[⌿]", 
    "A[☺]", "A[✗]", "A[I]", "✓ᵊ", "Nᵊ!", "⊕↓", "⊝!", "?cat(", "?p∈d(", 
    "?d∈p(", "?r∈p(", "E[ø]", "M[ø]", "»[ø]", "?d*",
    
    # --- Categorical Keywords ---
    "Type:N", "Type:S", "Type:R", "Type:B", "Type:Se", "PriceΔ", "DataΔ",
    
    # --- Catalog Identifiers (p01-p58, d01-d55, r01-r52) ---
    "p01", "p02", "p03", "p04", "p05", "p06", "p07", "p08",
    "p11", "p12", "p13", "p14", "p15", "p16", "p17", "p18",
    "p21", "p22", "p23", "p24", "p25", "p26", "p27", "p28",
    "p31", "p32", "p33", "p34", "p35", "p36", "p37", "p38",
    "p41", "p42", "p43", "p44", "p45", "p46", "p47", "p48",
    "p51", "p52", "p53", "p54", "p55", "p56", "p57", "p58",
    "d01", "d02", "d03", "d04", "d05", "d11", "d12", "d13", "d14", "d15",
    "d21", "d22", "d23", "d24", "d25", "d31", "d32", "d33", "d34", "d35",
    "d41", "d42", "d43", "d44", "d45", "d51", "d52", "d53", "d54", "d55",
    "r01", "r02", "r11", "r12", "r21", "r22", "r31", "r32", "r41", "r42", "r51", "r52"
]

# Clean duplicates
new_tokens = list(set(new_tokens))

def initialize_new_tokens(model, tokenizer, new_tokens):
    # Get the embedding layer
    embeddings = model.get_input_embeddings()
    # Get the LM head
    lm_head = model.get_output_embeddings()
    
    for token in new_tokens:
        token_id = tokenizer.convert_tokens_to_ids(token)
        # Simple heuristic: average the embeddings of the characters in the token
        # This gives the model a much better "starting point" than random noise
        sub_tokens = tokenizer.tokenize(token[1:] if token.startswith(' ') else token)
        sub_token_ids = tokenizer.convert_tokens_to_ids(sub_tokens)
        
        if len(sub_token_ids) > 0:
            with torch.no_grad():
                avg_emb = embeddings.weight[sub_token_ids].mean(dim=0)
                embeddings.weight[token_id] = avg_emb
                
                avg_lm = lm_head.weight[sub_token_ids].mean(dim=0)
                lm_head.weight[token_id] = avg_lm

# Call this after resizing but BEFORE training
tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))
# model.resize_output_embeddings(len(tokenizer))
initialize_new_tokens(model, tokenizer, new_tokens)

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    modules_to_save = ["embed_tokens", "lm_head"],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [6]:
######### data set for SFT

import json
from datasets import Dataset
import random

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

import json
import random
from datasets import Dataset

def prepare_data_set(file_name, tokenizer, max_seq_length):
    sft_data = []

    # 1. Load data
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                sft_data.append(json.loads(line))
    
    random.seed(42) 
    random.shuffle(sft_data)
    
    dataset = Dataset.from_dict({
        "messages": [item["messages"] for item in sft_data]
    })

    # 2. Format into ChatML strings
    def formatting_prompts_func(examples):
        output_texts = []
        for i in range(len(examples['messages'])):
            # add_generation_prompt=False is correct here because we want 
            # the full conversation (including the assistant's answer)
            text = tokenizer.apply_chat_template(
                examples['messages'][i], 
                tokenize=False, 
                add_generation_prompt=False
            )
            # Ensure the sequence ends with EOS
            if not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            output_texts.append(text)
        return { "text" : output_texts }

    # Map the formatting
    dataset = dataset.map(formatting_prompts_func, batched=True)

    # 3. Handle Tokenization
    # Note: We do NOT remove the 'text' column yet. 
    # SFTTrainer needs 'text' to handle completion-only masking.
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_seq_length,
            padding=False,
        )

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        # Keep 'text' column for Phase 2 collator!
        # Only remove 'messages' as it's a complex dict
        remove_columns=["messages"] 
    )
    
    print(f"Dataset ready with {len(tokenized_dataset)} rows.")
    return tokenized_dataset

In [9]:
#Phase 1 - 100 steps
from unsloth import UnslothTrainer, UnslothTrainingArguments
train_dataset = prepare_data_set("/mnt/data/training_data/final_training_data_250326.jsonl", tokenizer, max_seq_length)
sft_trainer_p1 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 5e-5,
        logging_steps = 5,
        save_steps = 50,
        save_total_limit=2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        
    ),
)


Map: 100%|██████████| 1122/1122 [00:03<00:00, 320.02 examples/s]


Dataset ready with 1122 rows.


In [10]:
sft_trainer_p1.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,122 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 484,855,808 of 2,261,587,456 (21.44% trained)


Step,Training Loss
5,4.478306
10,3.751503
15,2.711670
20,1.759745
25,1.039530
30,0.694273
35,0.581012
40,0.494771
45,0.427055
50,0.379251


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=100, training_loss=0.9493714869022369, metrics={'train_runtime': 379.0849, 'train_samples_per_second': 2.11, 'train_steps_per_second': 0.264, 'total_flos': 1.1069626309767168e+16, 'train_loss': 0.9493714869022369, 'epoch': 0.7130124777183601})

In [16]:
from transformers import DataCollatorForLanguageModeling
import torch

class UnifiedCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer, *args, mlm=False, **kwargs)
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        # Filter out 'text' strings to prevent the ValueError
        non_string_examples = []
        for e in examples:
            non_string_examples.append({k: v for k, v in e.items() if k != "text"})
            
        # Let the parent handle the tensor conversion/padding
        batch = super().torch_call(non_string_examples)
        
        # Apply the mask to 'labels'
        for i in range(len(batch["labels"])):
            curr_labels = batch["labels"][i]
            # Find sequence of token IDs
            for idx in range(len(curr_labels) - len(self.response_token_ids)):
                if curr_labels[idx : idx + len(self.response_token_ids)].tolist() == self.response_token_ids:
                    # Mask everything UP TO the end of the assistant header
                    batch["labels"][i][:idx + len(self.response_token_ids)] = -100
                    break
        return batch


# Initialize
collator = UnifiedCompletionCollator(
    response_template="<|im_start|>assistant\n", 
    tokenizer=tokenizer
)

In [17]:
# Test the collator on 1 sample
sample = [train_dataset[0]]
test_batch = collator.torch_call(sample)

# Check how many tokens are NOT masked (-100)
labels = test_batch["labels"][0]
unmasked_count = (labels != -100).sum().item()
total_count = len(labels)

print(f"Total tokens: {total_count}")
print(f"Unmasked tokens (Assistant only): {unmasked_count}")

# Decode the unmasked part to see if it's the <t> block
unmasked_tokens = labels[labels != -100]
print("What the model is learning:")
print(tokenizer.decode(unmasked_tokens))

Total tokens: 1280
Unmasked tokens (Assistant only): 34
What the model is learning:
<t>P[n|p45|75ᵊ|⊝]I[p45>p41+r41]E[p41✓,r41✗=✗]M[�]�[�]A[E]</t><|im_end|>
<|im_end|>


In [20]:
sft_trainer_p2 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,    
    data_collator=collator,
    dataset_text_field = "text",
    # formatting_func = format_sft_custom,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    # dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 5,
        save_steps = 25,
        save_total_limit=3,
        # eval_strategy="steps",  # Tell the trainer to evaluate periodically
        # eval_steps=10,                # Run evaluation every 10 steps
        # metric_for_best_model="loss", # Or "eval_loss" (lower is better)
        # greater_is_better=False,
        # load_best_model_at_end=True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        # completion_only_loss=True,
        # save_strategy="steps"
    ),
)


In [21]:
sft_trainer_p2.train()
# sft_trainer.evaluate()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,122 | Num Epochs = 1 | Total steps = 141
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 484,855,808 of 2,261,587,456 (21.44% trained)


Step,Training Loss
5,0.864442
10,0.910139
15,0.906876
20,0.766413
25,0.649664
30,0.642809
35,0.468539
40,0.365980
45,0.251175
50,0.284878


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and

TrainOutput(global_step=141, training_loss=0.3033050827100767, metrics={'train_runtime': 604.0294, 'train_samples_per_second': 1.858, 'train_steps_per_second': 0.233, 'total_flos': 1.5513469879504896e+16, 'train_loss': 0.3033050827100767, 'epoch': 1.0})

In [32]:
test_ids = tokenizer.encode("]»[", add_special_tokens=False)
print(f"Token IDs: {test_ids}")
print(f"Decoded: {[tokenizer.decode([i]) for i in test_ids]}")
# Check if your tokenizer actually knows the 'ø' symbol
print(f"Token ID for ø: {tokenizer.encode('ø', add_special_tokens=False)}")

Token IDs: [60, 151740]
Decoded: [']', '�[']
Token ID for ø: [180]


In [ ]:
model.save_pretrained("/mnt/data/models/amaiz_telecom_model_4b_1003_V_adap")
tokenizer.save_pretrained("/mnt/data/models/amaiz_telecom_model_4b_1003_V_adap")

In [ ]:
model.save_pretrained_gguf("/mnt/data/models/amaiz_telecom_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
for name, param in model.named_parameters():
    if "lora" in name:
        if not param.requires_grad:
            print(f"🚨 WARNING: {name} is FROZEN! Training will not work.")
        else:
            print(f"✅ {name} is trainable.")
        break